# FlowDraft — Train YOLO on P&ID Symbols (Kaggle)

Train a symbol detector for `POST /parse`. Dataset: **[P&ID Symbols for Computer Vision](https://www.kaggle.com/datasets/mollous/p-id-symbols-for-computer-vision)** (Mol Lous, 203 classes, YOLO format).

## Before you run

1. **Kaggle → New Notebook** (or open this file in Kaggle).
2. **Settings → Accelerator → GPU** (T4 is enough).
3. **Add Input → search "P&ID Symbols"** → add *P&ID Symbols for Computer Vision*.
4. **Run all cells** (Runtime → Run All).

## After training — download weights

1. Open the notebook **Output** tab (right sidebar).
2. Download **`yolov8n_pid.pt`** (and optionally `best.pt` — same file).
3. On your machine, copy to the repo:
   ```
   eurotech/models/yolov8n.pt
   ```
4. Smoke test:
   ```powershell
   python src/04_build_graph.py data/raw_diagrams/images__train__113.jpg --out data/parsed_live.json
   ```
   Expect `equipment.pump.niso`, `valve.*` — **not** `boat`.

> **Do not** use COCO `yolov8n.pt` from the internet. The API refuses COCO weights.

## Optional: better mAP (yolov8s, 80 epochs)

In the training cell, set `MODEL = "yolov8s.pt"` and `EPOCHS = 80`. Re-download and replace `models/yolov8n.pt`.

In [ ]:
!pip install -q ultralytics pyyaml

In [ ]:
from pathlib import Path

INPUT = Path("/kaggle/input")

print("Folders in /kaggle/input:")
for p in sorted(INPUT.iterdir()):
    print(" ", p.name)

# Find dataset root: folder that contains data.yaml
DATA_ROOT = None
for folder in INPUT.iterdir():
    if not folder.is_dir():
        continue
    if (folder / "data.yaml").exists():
        DATA_ROOT = folder
        break

if DATA_ROOT is None:
    for yaml_file in INPUT.glob("*/data.yaml"):
        DATA_ROOT = yaml_file.parent
        break

# If still None, uncomment and set the folder name printed above:
# DATA_ROOT = Path("/kaggle/input/p-id-symbols-for-computer-vision")

if DATA_ROOT is None:
    raise SystemExit(
        "Could not find data.yaml. Add the P&ID Symbols dataset (Add Input), "
        "re-run, or set DATA_ROOT manually."
    )

print("Using DATA_ROOT =", DATA_ROOT)
print("data.yaml:", (DATA_ROOT / "data.yaml").exists())
print("labels/train:", (DATA_ROOT / "labels" / "train").exists())
print("images/train:", (DATA_ROOT / "images" / "train").exists())

In [ ]:
# Labels must be 5 numbers per line (class xc yc w h). NOT 9 (OBB).
label_dir = DATA_ROOT / "labels" / "train"
label_files = list(label_dir.glob("*.txt")) or list((DATA_ROOT / "labels").rglob("*.txt"))

print("Label files:", len(label_files))
sample = label_files[0]
first_lines = [ln.strip() for ln in open(sample).read().splitlines() if ln.strip()][:3]
print("Sample:", sample.name)
for i, line in enumerate(first_lines):
    parts = line.split()
    print(f"  line {i} → {len(parts)} values:", parts[:6])
    if len(parts) == 5:
        print("  → OK: use yolov8n.pt (normal detection)")
    elif len(parts) >= 8:
        raise SystemExit("OBB labels detected — use yolov8n-obb.pt instead")

In [ ]:
# Ultralytics needs absolute paths on Kaggle
import yaml

with open(DATA_ROOT / "data.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

for key in ("train", "val", "test"):
    if key in cfg and cfg[key]:
        p = Path(cfg[key])
        if not p.is_absolute():
            p = DATA_ROOT / cfg[key]
        cfg[key] = str(p.resolve())

yaml_path = Path("/kaggle/working/pid_data.yaml")
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("Wrote", yaml_path)
print("nc (classes):", cfg.get("nc"))
print("train:", cfg.get("train"))
print("val:", cfg.get("val"))

In [ ]:
from ultralytics import YOLO

# --- config: change for optional retrain ---
MODEL = "yolov8n.pt"   # optional upgrade: "yolov8s.pt"
EPOCHS = 30            # optional upgrade: 80
IMGSZ = 640
BATCH = 16

model = YOLO(MODEL)

results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=10,
    project="/kaggle/working/runs/detect",
    name="train",
    exist_ok=True,
)

BEST = "/kaggle/working/runs/detect/train/weights/best.pt"
print("Done.")
print("save_dir:", results.save_dir)
print("best:", BEST)

In [ ]:
# Validation metrics (mAP50 overall ~12% on 203 classes with yolov8n/30ep is normal)
m = YOLO(BEST)
metrics = m.val(data=str(yaml_path))
print(metrics)
print("\nStrong classes to demo: equipment.pump.niso, valve.control.iso")

In [ ]:
# Export for download — appears in Kaggle Output tab
import shutil
from pathlib import Path

out = Path("/kaggle/working/yolov8n_pid.pt")
shutil.copy(BEST, out)
size_mb = out.stat().st_size / (1024 * 1024)
print(f"Saved {out} ({size_mb:.1f} MB)")
print()
print("NEXT STEPS:")
print("  1. Kaggle → Output tab → download yolov8n_pid.pt")
print("  2. Copy to your repo as models/yolov8n.pt")
print("  3. python src/04_build_graph.py data/raw_diagrams/images__train__113.jpg")

In [ ]:
# Quick sanity check: class names should be P&ID, not COCO
names = list(m.names.values())
print("num classes:", len(names))
print("sample:", names[:5])
assert "boat" not in names and "person" not in names, "COCO weights — training failed?"
assert len(names) >= 200, f"expected ~203 classes, got {len(names)}"
print("OK — P&ID weights ready for download")